# Arabic Sign Language - YOLOv5 Training
Train a YOLOv5 model to detect Arabic Sign Language letters.

**Features:**
- Saves checkpoints to Google Drive (survives runtime disconnects)
- Can resume training from last checkpoint
- Dataset stays on Drive, no re-uploading needed

**Before you start:**
1. Upload the dataset folder to Google Drive at: `My Drive/ArSL_Project/datasets/`
2. Set runtime to GPU: Runtime > Change runtime type > T4 GPU

## Step 1: Mount Google Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ArSL_Project'
DATASET_DIR = os.path.join(PROJECT_DIR, 'datasets')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!pip install ultralytics -q

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2: Verify Dataset & Create data.yaml

In [ ]:
import yaml

train_imgs = os.path.join(DATASET_DIR, 'train', 'images')
val_imgs = os.path.join(DATASET_DIR, 'valid', 'images')

for path, name in [(DATASET_DIR + '/train/images', 'Train images'),
                    (DATASET_DIR + '/train/labels', 'Train labels'),
                    (DATASET_DIR + '/valid/images', 'Val images'),
                    (DATASET_DIR + '/valid/labels', 'Val labels')]:
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    status = 'OK' if count > 0 else 'MISSING!'
    print(f'{name}: {count} files [{status}]')

ARABIC_CLASSES = [
    'ain', 'al', 'aleff', 'bb', 'dal', 'dha', 'dhad', 'fa',
    'gaaf', 'ghain', 'ha', 'haa', 'jeem', 'kaaf', 'khaa', 'la',
    'laam', 'meem', 'nun', 'ra', 'saad', 'seen', 'sheen', 'ta',
    'taa', 'thaa', 'thal', 'toot', 'waw', 'ya', 'yaa', 'zay'
]

yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump({
        'train': train_imgs,
        'val': val_imgs,
        'nc': 32,
        'names': ARABIC_CLASSES
    }, f, default_flow_style=False, allow_unicode=True)

print(f'\ndata.yaml ready!')

## Step 3: Train YOLOv5 (saves to Drive — resumable!)

Training saves directly to Google Drive so if runtime disconnects,
your progress is safe. Just re-run this cell to resume.

In [ ]:
from ultralytics import YOLO

# Check if there's a previous checkpoint to resume from
resume_path = os.path.join(CHECKPOINT_DIR, 'weights', 'last.pt')

if os.path.exists(resume_path):
    print('Found previous checkpoint! Resuming training...')
    print(f'Checkpoint: {resume_path}')
    model = YOLO(resume_path)
    results = model.train(resume=True)
else:
    print('Starting fresh training...')
    model = YOLO('yolov5s.pt')
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        patience=20,
        save=True,
        save_period=5,       # save checkpoint every 5 epochs
        project=CHECKPOINT_DIR,  # save directly to Drive!
        name='',
        exist_ok=True,
    )

print('\nTraining complete!')

## Step 4: Evaluate Model

In [ ]:
best_pt = os.path.join(CHECKPOINT_DIR, 'weights', 'best.pt')
best_model = YOLO(best_pt)
metrics = best_model.val(data=yaml_path)
print(f'\nmAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## Step 5: Export & Download

Download `arabic_sign_best.pt` and place it in your local project:
```
models/arabic_sign_best.pt
```

In [ ]:
import shutil

best_pt = os.path.join(CHECKPOINT_DIR, 'weights', 'best.pt')

# Copy with a clean name
final_dest = os.path.join(PROJECT_DIR, 'arabic_sign_best.pt')
shutil.copy2(best_pt, final_dest)
print(f'Model saved to Drive: {final_dest}')

# Export ONNX
best_model.export(format='onnx')
onnx_src = best_pt.replace('.pt', '.onnx')
if os.path.exists(onnx_src):
    onnx_dest = os.path.join(PROJECT_DIR, 'arabic_sign_best.onnx')
    shutil.copy2(onnx_src, onnx_dest)
    print(f'ONNX saved to Drive: {onnx_dest}')

print(f'\nDone! Download arabic_sign_best.pt from your Google Drive.')

In [ ]:
# Optional: Download directly to your browser
from google.colab import files
files.download(final_dest)